# Kaggle ARC Prize 2026: SOTA Programmatic Synthesis & Test-Time Search Engine
### Accelerated on NVIDIA A100 GPU (40GB/80GB SXM4) with Unsloth & vLLM

This notebook implements an end-to-end competitive pipeline for ARC Prize 2026:
1. **High-Throughput Environment:** A100 GPU setup, FlashAttention-2, Unsloth, and vLLM.
2. **Data Augmentations:** Dihedral D4 symmetries (rotations/reflections) and color space permutations with exact mathematical inversions.
3. **Reasoning / Coder Model:** Qwen2.5-Coder-7B-Instruct with LoRA programmatic transformation synthesis.
4. **Test-Time Search & Verification:** Generates candidate Python transformation programs, verifies them in a sandboxed interpreter on training pairs, and retains verified programs.
5. **Ensembling & Submission:** Top-2 submission extraction formatted for Kaggle competition submission.

In [ ]:
# 1. Verify NVIDIA A100 GPU and System Resources
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU")

In [ ]:
# 2. High-Throughput Tooling Installation
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install vllm transformers datasets kaggle

In [ ]:
# 3. Mount Google Drive for Checkpoints & Persistence
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ARC_PRIZE_2026'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive directory ready at: {DRIVE_DIR}")

In [ ]:
# 4. Setup Kaggle & Download ARC Competitions
import json
import os

# Kaggle token setup
# Ensure kaggle.json is present or pass KAGGLE_API_TOKEN environment variable
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ 2>/dev/null || true
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null || true

!mkdir -p /content/data/arc_agi_2
!mkdir -p /content/data/arc_agi_3

# Download official ARC datasets
!kaggle competitions download -c arc-prize-2026-arc-agi-2 -p /content/data/arc_agi_2 --force || true
!kaggle competitions download -c arc-prize-2026-arc-agi-3 -p /content/data/arc_agi_3 --force || true

# Also clone the canonical ARC-AGI benchmark tasks
!git clone --depth 1 https://github.com/fchollet/ARC-AGI.git /content/data/fchollet_arc_agi || true
print("Datasets initialized.")

In [ ]:
# 5. Data Augmentation Engine (D4 Dihedral Group & Color Space Permutations)
import copy
import random
from typing import Dict, Any, List, Tuple
import numpy as np

D4_OPERATIONS = ["identity", "rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose", "anti_transpose"]

def apply_d4(grid: np.ndarray, op: str) -> np.ndarray:
    if op == "identity": return grid.copy()
    elif op == "rot90": return np.rot90(grid, -1).copy()
    elif op == "rot180": return np.rot90(grid, 2).copy()
    elif op == "rot270": return np.rot90(grid, 1).copy()
    elif op == "flip_h": return np.fliplr(grid).copy()
    elif op == "flip_v": return np.flipud(grid).copy()
    elif op == "transpose": return grid.T.copy()
    elif op == "anti_transpose": return np.rot90(grid.T, 2).copy()
    raise ValueError(f"Unknown op: {op}")

def invert_d4(grid: np.ndarray, op: str) -> np.ndarray:
    if op == "identity": return grid.copy()
    elif op == "rot90": return np.rot90(grid, 1).copy()
    elif op == "rot180": return np.rot90(grid, 2).copy()
    elif op == "rot270": return np.rot90(grid, -1).copy()
    elif op == "flip_h": return np.fliplr(grid).copy()
    elif op == "flip_v": return np.flipud(grid).copy()
    elif op == "transpose": return grid.T.copy()
    elif op == "anti_transpose": return np.rot90(grid.T, 2).copy()
    raise ValueError(f"Unknown op: {op}")

def augment_task_d4(task: Dict[str, Any], op: str) -> Dict[str, Any]:
    new_task = copy.deepcopy(task)
    for pair in new_task.get("train", []):
        pair["input"] = apply_d4(np.array(pair["input"]), op).tolist()
        pair["output"] = apply_d4(np.array(pair["output"]), op).tolist()
    for pair in new_task.get("test", []):
        pair["input"] = apply_d4(np.array(pair["input"]), op).tolist()
        if "output" in pair:
            pair["output"] = apply_d4(np.array(pair["output"]), op).tolist()
    return new_task

print("D4 Augmentation Engine ready.")

In [ ]:
# 6. Sandboxed Execution & Verification Engine
import ast
import re
import concurrent.futures

def extract_code_block(text: str) -> str:
    match = re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match: return match.group(1).strip()
    match = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
    if match: return match.group(1).strip()
    return text.strip()

def _safe_import(name, *args, **kwargs):
    allowed = {"numpy", "np", "math", "collections", "itertools", "scipy", "copy", "re"}
    top = name.split(".")[0]
    if top not in allowed:
        raise ImportError(f"Import not allowed: {name}")
    return __import__(name, *args, **kwargs)

def _run_isolated_transform(code_str: str, input_grid: list):
    try:
        builtins_dict = {
            k: __builtins__[k] for k in [
                "range", "len", "enumerate", "zip", "map", "filter",
                "list", "dict", "set", "tuple", "int", "float", "bool",
                "str", "min", "max", "sum", "abs", "round", "sorted",
                "any", "all", "print", "Exception", "ValueError", "TypeError",
                "IndexError", "KeyError", "isinstance", "issubclass"
            ] if (isinstance(__builtins__, dict) and k in __builtins__) or hasattr(__builtins__, k)
        }
        builtins_dict["__import__"] = _safe_import
        scope = {"np": np, "numpy": np, "__builtins__": builtins_dict}
        exec(code_str, scope)
        if "transform" not in scope or not callable(scope["transform"]):
            return False, None, "Function 'transform' not found"
        out = scope["transform"](copy.deepcopy(input_grid))
        if isinstance(out, np.ndarray): out = out.tolist()
        elif isinstance(out, list):
            out = [list(r) if isinstance(r, (list, tuple, np.ndarray)) else r for r in out]
        return True, out, ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {str(e)}"

def execute_with_timeout(code_str: str, input_grid: list, timeout_sec: float = 2.0):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        fut = executor.submit(_run_isolated_transform, code_str, input_grid)
        try:
            return fut.result(timeout=timeout_sec)
        except concurrent.futures.TimeoutError:
            return False, None, "Timeout exceeded"
        except Exception as e:
            return False, None, str(e)

def grids_equal(g1, g2):
    if not isinstance(g1, list) or not isinstance(g2, list) or len(g1) != len(g2):
        return False
    return all(list(r1) == list(r2) for r1, r2 in zip(g1, g2))

def evaluate_program(code_str: str, task: Dict[str, Any]):
    cleaned = extract_code_block(code_str)
    try:
        ast.parse(cleaned)
    except SyntaxError:
        return False, []
    # Verify on all train pairs
    for pair in task.get("train", []):
        ok, pred, _ = execute_with_timeout(cleaned, pair["input"])
        if not ok or not grids_equal(pred, pair["output"]):
            return False, []
    # If all train match, predict test queries
    test_preds = []
    for pair in task.get("test", []):
        ok, pred, _ = execute_with_timeout(cleaned, pair["input"])
        test_preds.append(pred if ok else None)
    return True, test_preds

print("Sandbox verification engine ready.")

In [ ]:
# 7. Load Qwen2.5-Coder-7B-Instruct with Unsloth on A100
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Configure Fast LoRA Adapters for Programmatic ARC Reasoning
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42
)
print("Qwen2.5-Coder-7B-Instruct LoRA Model Loaded Successfully!")

In [ ]:
# 8. Test-Time Search Engine with D4 TTA & Program Ensembling
from collections import Counter

FastLanguageModel.for_inference(model)

def format_arc_prompt(task: Dict[str, Any]) -> str:
    body = ["You are an expert ARC solver. Write a Python function `def transform(grid: list[list[int]]) -> list[list[int]]:` to solve this task.\n"]
    for i, pair in enumerate(task.get("train", [])):
        body.append(f"Example {i+1} Input: {pair['input']}\nExample {i+1} Output: {pair['output']}\n")
    for i, pair in enumerate(task.get("test", [])):
        body.append(f"Test Query {i+1} Input: {pair['input']}\n")
    body.append("Write the Python function wrapped in ```python ... ```:")
    return "\n".join(body)

def generate_candidate_programs(prompt: str, num_samples: int = 8, temperature: float = 0.7) -> List[str]:
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=temperature,
        top_p=0.95,
        num_return_sequences=min(num_samples, 4),
        do_sample=True
    )
    return [tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True) for out in outputs]

def solve_arc_task_with_search(task: Dict[str, Any], samples_per_op: int = 4) -> List[Dict[str, Any]]:
    num_test = len(task.get("test", []))
    pred_counters = [Counter() for _ in range(num_test)]
    
    # D4 TTA Search
    for op in ["identity", "rot90", "flip_h", "flip_v"]:
        aug_task = augment_task_d4(task, op)
        prompt = format_arc_prompt(aug_task)
        candidates = generate_candidate_programs(prompt, num_samples=samples_per_op)
        for code in candidates:
            solved, test_preds = evaluate_program(code, aug_task)
            if solved:
                for t_idx, p in enumerate(test_preds):
                    if p is not None:
                        restored = invert_d4(np.array(p), op).tolist() if op != "identity" else p
                        pred_counters[t_idx][json.dumps(restored)] += 1
                        
    # Select top-2 submission candidates
    results = []
    for t_idx in range(num_test):
        top = pred_counters[t_idx].most_common(2)
        fallback = task["test"][t_idx]["input"]
        p1 = json.loads(top[0][0]) if len(top) > 0 else fallback
        p2 = json.loads(top[1][0]) if len(top) > 1 else p1
        results.append({"attempt_1": p1, "attempt_2": p2})
    return results

print("Search engine initialized and ready.")

In [ ]:
# 9. Verification & Submission Generation
import glob

# Load evaluation sample tasks
eval_files = glob.glob("/content/data/fchollet_arc_agi/data/training/*.json")[:10]
print(f"Benchmarking across {len(eval_files)} sample tasks...")

submission = {}
for fpath in eval_files:
    task_id = os.path.splitext(os.path.basename(fpath))[0]
    with open(fpath) as f:
        task = json.load(f)
    submission[task_id] = solve_arc_task_with_search(task, samples_per_op=2)
    print(f"Task {task_id} processed.")

sub_path = os.path.join(DRIVE_DIR, "submission_arc_2026.json")
with open(sub_path, "w") as f:
    json.dump(submission, f)
print(f"Submission written to {sub_path}")

# Submit to Kaggle
!kaggle competitions submit -c arc-prize-2026-arc-agi-2 -f {sub_path} -m "Unsloth Qwen2.5-Coder A100 Test-Time Search" || true